In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    root='data',
    train=True,
    transform=transform,
    download=True
)

subset = torch.utils.data.Subset(train_dataset, range(5000))
loader = DataLoader(subset, batch_size=32, shuffle=True)

In [ ]:
import torch.nn as nn

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.fc = nn.Linear(16 * 14 * 14, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)

        x = torch.relu(self.conv2(x))
        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

model = SmallCNN()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    running_loss = 0.0

    for images, labels in loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

model.eval()

with torch.no_grad():
    sample_images, sample_labels = next(iter(loader))
    preds = model(sample_images).argmax(dim=1)

plt.figure(figsize=(8, 4))

for i in range(8):
    plt.subplot(2, 4, i+1)
    plt.imshow(sample_images[i].squeeze(), cmap='gray')
    plt.title(f"pred: {preds[i].item()}")
    plt.axis('off')

plt.show()

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

t = np.linspace(0, 100, 1000)
data = np.sin(t).astype(np.float32)

window = 20

class SineDataset(Dataset):
    def __init__(self, data, window):
        self.data = data
        self.window = window

    def __len__(self):
        return len(self.data) - self.window

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.window]
        y = self.data[idx+self.window]
        return torch.tensor(x).unsqueeze(1), torch.tensor(y)

dataset = SineDataset(data, window)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
import torch.nn as nn

class LSTMForecaster(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=16, batch_first=True)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        y = self.fc(last)
        return y.squeeze(1)

model = LSTMForecaster()

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 25

for epoch in range(num_epochs):
    total_loss = 0.0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()

        preds = model(X_batch)
        loss = criterion(preds, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

model.eval()

test_input = data[200:200+window]
current = torch.tensor(test_input).unsqueeze(0).unsqueeze(2)

preds = []

for _ in range(50):
    with torch.no_grad():
        y_hat = model(current)
    preds.append(y_hat.item())

    new_seq = torch.cat([current[:, 1:, :], y_hat.view(1, 1, 1)], dim=1)
    current = new_seq

plt.plot(range(20), test_input, label='Input Sequence')
plt.plot(range(20, 70), preds, label='Predicted')
plt.legend()
plt.title("LSTM Time-Series Forecast")
plt.show()